In [151]:
# magics: ensures that any changes to the modules loaded below will be re-loaded automatically
%load_ext autoreload
%autoreload 2

# load general packages
import os
os.chdir('/Users/jacobaspnissen/Desktop/Økonomi/Dynamic programming/Termpaper/termpaper_dynprog')
# os.chdir('/Users/albertolsen/Documents/Stud.Polit./9. semester/Dynamic Programming/Exam/termpaper_dynprog')

import numpy as np
import time
import copy
import pandas as pd
import pyreadr
import pickle
import xarray as xr  # Ensure xarray is installed
import statsmodels.api as sm



import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from matplotlib import cm
plt.style.use('seaborn-v0_8-whitegrid')

# load modules related EGM
#import tools 
#from model_exante import model_bufferstock
#import estimate_exante as estimate

# load modules related to NFXP
from model_retirement import retirement
from Solve_NFXP import solve_NFXP
import estimate_NFXP as estimate

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Import Data 

In [152]:
# Clean data
cleaned_lines = []
with open("KPS Data/sparadata2.txt", "r") as file:
    for line in file:
        # Replace multiple spaces with a single tab
        cleaned_line = " ".join(line.split())  # Normalize spaces
        cleaned_lines.append(cleaned_line)

# Save the cleaned file
with open("KPS Data/cleaned_sparadata.txt", "w") as cleaned_file:
    cleaned_file.write("\n".join(cleaned_lines))

In [153]:
datapath = "KPS Data/cleaned_sparadata.txt"
sparadata = np.genfromtxt(open(datapath, "rb"), delimiter=" ", skip_header=0, dtype=None, encoding=None)


- index   is an identifier for the observation (each individual and year)
- id2    	is an identifier for the individual
- sex    	1=man 2=woman
- year   	of observation
- age    	of the individual for the given year
- married	marital status
- retire	yes=1, no=0
- income	1/1000 SEK. This number is rounded (no decimal digits) due to
	confidentiality reasons
- atp	is the "average pension points" used in the paper

In [154]:
data = pd.read_csv(datapath, sep=" ", header=0, encoding='utf-8')
data = data.drop(columns="index")
# create data['atp_1'] as the atp for the same individual in the next period using id to check if individual is the same and year to check if it is the next period
data['atp_1'] = data.groupby('id')['atp'].shift(-1)  
data['atp_1'] = data['atp_1'].fillna(data['atp'])  # Fill NaN values with corresponding 'atp' value if next period is not available
#data['atp'] = data['atp'].astype(float)
#data['atp_1'] = data['atp_1'].astype(float)
data['log_atp'] = np.log(data['atp'])
data['age_squared'] = data['age'] ** 2
data['income_1'] = data.groupby('id')['income'].shift(-1)  # Shift income to get next period's income
data['income_1'] = data['income_1'].fillna(0)  # Fill NaN values with corresponding 'income' value if next period is not available
data['log_income'] = np.log(data['income'])
data

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


,id,sex,year,age,married,ret,income,atp,atp_1,log_atp,age_squared,income_1,log_income
0,3,1,83,52,1,0.0,179,4.636000,4.636000,1.533852,2704,188.0,5.187386
1,3,1,84,53,1,0.0,188,4.636000,4.636000,1.533852,2809,186.0,5.236442
2,3,1,85,54,1,0.0,186,4.636000,4.636000,1.533852,2916,189.0,5.225747
3,3,1,86,55,1,0.0,189,4.636000,4.640667,1.533852,3025,196.0,5.241747
4,3,1,87,56,1,0.0,196,4.640667,4.658000,1.534858,3136,208.0,5.278115
...,...,...,...,...,...,...,...,...,...,...,...,...,...
51366,44094,2,92,52,1,0.0,154,2.803333,2.875333,1.030809,2704,154.0,5.036953
51367,44094,2,93,53,1,0.0,154,2.875333,2.944667,1.056169,2809,154.0,5.036953
51368,44094,2,94,54,0,0.0,154,2.944667,3.009333,1.079996,2916,154.0,5.036953
51369,44094,2,95,55,0,0.0,154,3.009333,3.064667,1.101719,3025,154.0,5.036953


## Predicting atp transitions (deterministic)

In [155]:
# State variables: age, wage, average atp points, retirement age and marital status
# Regularize to avoid log(0)
epsilon = 1e-4
data2 = data[data['atp'] > 0]  # Remove 0s to avoid -inf
data2 = data2[data2['atp_1'] > 0]  # Remove 0s in atp_1 to avoid -inf


# Construct regression variables
X = sm.add_constant(data2[['log_atp', 'age', 'age_squared']])

y = np.log(data2['atp_1'])

# Estimate OLS
model = sm.OLS(y, X).fit()
print(model.summary())

# Extract coefficients and residual variance
gamma = model.params
sigma2 = model.mse_resid

# Predict next period ATP points
data['atp_next'] = np.exp(model.predict(sm.add_constant(data[['log_atp', 'age', 'age_squared']])))
data['atp_next'] = data['atp_next'].round(3)  # Adjust the number of decimals as needed

                            OLS Regression Results                            
Dep. Variable:                  atp_1   R-squared:                       0.997
Model:                            OLS   Adj. R-squared:                  0.997
Method:                 Least Squares   F-statistic:                 5.470e+06
Date:                Sat, 02 Aug 2025   Prob (F-statistic):               0.00
Time:                        15:34:21   Log-Likelihood:             1.1707e+05
No. Observations:               51370   AIC:                        -2.341e+05
Df Residuals:                   51366   BIC:                        -2.341e+05
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                  coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------
const          -0.0418      0.020     -2.097      

In [156]:
data['atp_next'] = np.exp(sm.add_constant(data[['log_atp', 'age', 'age_squared']]).dot(gamma)+sigma2/2)  # Predict next period ATP points with noise

In [157]:
# Define the number of bins
num_bins = 20  # Number of bins for income

# Define income bins
atp_bins = np.linspace(data['atp'].min(), data['atp_next'].max(), num_bins + 1)

# Define bin labels (0 to num_bins - 1)
bin_labels = range(num_bins)

# Initialize a dictionary to store transition matrices for each age
#transition_matrices = {}


    
# Bin income and income_next into the defined bins
#data['atp_bin'] = pd.cut(data['atp'], bins=atp_bins, labels=bin_labels, include_lowest=True)
#data['atp_next_bin'] = pd.cut(data['atp_next'], bins=atp_bins, labels=bin_labels, include_lowest=True)

# Use quantile-based bins for ATP
data['atp_bin'] = pd.qcut(data['atp'], q=num_bins, labels=False, duplicates='drop')
data['atp_next_bin'] = pd.qcut(data['atp_next'], q=num_bins, labels=False, duplicates='drop')


# Create a transition matrix for the current age
atp_transition_matrix = pd.crosstab(data['atp_bin'], data['atp_next_bin'], normalize='index')

# Reindex the transition matrix to ensure all bins are included
atp_transition_matrix = atp_transition_matrix.reindex(index=bin_labels, columns=bin_labels, fill_value=0)

# Optionally enforce single-column transitions
atp_transition_matrix = atp_transition_matrix.apply(lambda row: (row == row.max()).astype(float), axis=1)


# Example: Display the transition matrix for age 50
#print("Transition matrix for age 50:")
#print(transition_matrices[50])

# Example: Display the transition matrix for age 62
print("ATP Transition Matrix")
print(atp_transition_matrix)

ATP Transition Matrix
atp_next_bin   0    1    2    3    4    5    6    7    8    9    10   11   12  \
atp_bin                                                                         
0             1.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0   
1             0.0  1.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0   
2             0.0  0.0  1.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0   
3             0.0  0.0  0.0  1.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0   
4             0.0  0.0  0.0  0.0  1.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0   
5             0.0  0.0  0.0  0.0  0.0  1.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0   
6             0.0  0.0  0.0  0.0  0.0  0.0  1.0  0.0  0.0  0.0  0.0  0.0  0.0   
7             0.0  0.0  0.0  0.0  0.0  0.0  0.0  1.0  0.0  0.0  0.0  0.0  0.0   
8             0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  1.0  0.0  0.0  0.0  0.0   
9             0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  1.0  0.0  0.0  0.0   
10    

## The income transitions (empirical transition frequencies)

In [158]:
# State variables: age, wage, average atp points, retirement age and marital status
# Regularize to avoid log(0)
epsilon = 1e-4
data3 = data[data['income_1'] > 0]  # Remove 0s to avoid -inf



# Construct regression variables
X2 = sm.add_constant(data3[['log_income', 'age', 'age_squared']])

y2 = np.log(data3['income_1'])

# Estimate OLS
model2 = sm.OLS(y2, X2).fit()
print(model2.summary())

# Extract coefficients and residual variance
alpha = model2.params
zeta2 = model2.mse_resid

data['income_next'] = np.exp(model2.predict(sm.add_constant(data[['log_income', 'age', 'age_squared']])))

                            OLS Regression Results                            
Dep. Variable:               income_1   R-squared:                       0.809
Model:                            OLS   Adj. R-squared:                  0.809
Method:                 Least Squares   F-statistic:                 6.590e+04
Date:                Sat, 02 Aug 2025   Prob (F-statistic):               0.00
Time:                        15:34:22   Log-Likelihood:                 18133.
No. Observations:               46734   AIC:                        -3.626e+04
Df Residuals:                   46730   BIC:                        -3.622e+04
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                  coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------
const           0.2376      0.153      1.551      

In [159]:
data

,id,sex,year,age,married,ret,income,atp,atp_1,log_atp,age_squared,income_1,log_income,atp_next,atp_bin,atp_next_bin,income_next
0,3,1,83,52,1,0.0,179,4.636000,4.636000,1.533852,2704,188.0,5.187386,4.686988,14,14,179.261481
1,3,1,84,53,1,0.0,188,4.636000,4.636000,1.533852,2809,186.0,5.236442,4.684262,14,14,187.242677
2,3,1,85,54,1,0.0,186,4.636000,4.636000,1.533852,2916,189.0,5.225747,4.681128,14,14,185.438454
3,3,1,86,55,1,0.0,189,4.636000,4.640667,1.533852,3025,196.0,5.241747,4.677585,14,14,188.003908
4,3,1,87,56,1,0.0,196,4.640667,4.658000,1.534858,3136,208.0,5.278115,4.678224,14,14,194.029929
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
51366,44094,2,92,52,1,0.0,154,2.803333,2.875333,1.030809,2704,154.0,5.036953,2.869850,4,4,156.872835
51367,44094,2,93,53,1,0.0,154,2.875333,2.944667,1.056169,2809,154.0,5.036953,2.939991,4,4,156.881475
51368,44094,2,94,54,0,0.0,154,2.944667,3.009333,1.079996,2916,154.0,5.036953,3.007086,4,4,156.850498
51369,44094,2,95,55,0,0.0,154,3.009333,3.064667,1.101719,3025,154.0,5.036953,3.069140,5,5,156.779928


In [160]:
# Define the age range
age_range = range(50, 70)

# Define income bins
income_bins = np.linspace(data['income'].min(), data['income'].max(), num_bins + 1)
# Check the bins
print("Income bins:")
print(income_bins)

# Define bin labels (0 to num_bins - 1)
bin_labels = range(num_bins)

# Initialize a dictionary to store transition matrices for each age
inc_transition_matrices = {}

 # Use quantile-based bins
data['income_bin'] = pd.qcut(data['income'], q=num_bins, labels=False)
data['income_next_bin'] = pd.qcut(data['income_1'], q=num_bins, labels=False, duplicates='drop')

# Loop through each age in the range
for age in age_range:
    # Filter data for the current age
    age_data = data[data['age'] == age]
    
    # Bin income and income_next into the defined bins
    #age_data['income_bin'] = pd.cut(age_data['income'], bins=income_bins, labels=bin_labels, include_lowest=True)
    #age_data['income_next_bin'] = pd.cut(age_data['income_1'], bins=income_bins, labels=bin_labels, include_lowest=True)

    # Create a transition matrix for the current age
    inc_transition_matrix = pd.crosstab(age_data['income_bin'], age_data['income_next_bin'], normalize='index')
    
    # Reindex the transition matrix to ensure all bins are included
    inc_transition_matrix = inc_transition_matrix.reindex(index=bin_labels, columns=bin_labels, fill_value=0)
    
    # Store the transition matrix
    inc_transition_matrices[age] = inc_transition_matrix

# Example: Display the transition matrix for age 50
#print("Transition matrix for age 50:")
#print(transition_matrices[50])

# Example: Display the transition matrix for age 62
print("Transition matrix for age 55:")
print(inc_transition_matrices[66])

Income bins:
[1.0000e+00 8.0500e+01 1.6000e+02 2.3950e+02 3.1900e+02 3.9850e+02
 4.7800e+02 5.5750e+02 6.3700e+02 7.1650e+02 7.9600e+02 8.7550e+02
 9.5500e+02 1.0345e+03 1.1140e+03 1.1935e+03 1.2730e+03 1.3525e+03
 1.4320e+03 1.5115e+03 1.5910e+03]
Transition matrix for age 55:
income_next_bin        0    1    2         3         4         5         6   \
income_bin                                                                    
0                0.800000  0.2  0.0  0.000000  0.000000  0.000000  0.000000   
1                0.600000  0.2  0.2  0.000000  0.000000  0.000000  0.000000   
2                0.375000  0.0  0.5  0.125000  0.000000  0.000000  0.000000   
3                0.333333  0.0  0.0  0.333333  0.333333  0.000000  0.000000   
4                0.333333  0.0  0.0  0.000000  0.333333  0.333333  0.000000   
5                0.285714  0.0  0.0  0.000000  0.000000  0.428571  0.142857   
6                0.400000  0.0  0.0  0.000000  0.000000  0.000000  0.600000   
7         

In [161]:
p = {}
for age in age_range:
    transition_matrix = np.kron(atp_transition_matrix, inc_transition_matrices[age])
    p[age] = transition_matrix
p[60]
# check if sum of each row is 1
for age in age_range:
    if not np.allclose(p[age].sum(axis=1), 1):
        print(f"Row sums for age {age} do not equal 1")
    else:
        print(f"Row sums for age {age} equal 1")

Row sums for age 50 equal 1
Row sums for age 51 equal 1
Row sums for age 52 equal 1
Row sums for age 53 equal 1
Row sums for age 54 equal 1
Row sums for age 55 equal 1
Row sums for age 56 equal 1
Row sums for age 57 equal 1
Row sums for age 58 equal 1
Row sums for age 59 equal 1
Row sums for age 60 equal 1
Row sums for age 61 equal 1
Row sums for age 62 equal 1
Row sums for age 63 equal 1
Row sums for age 64 equal 1
Row sums for age 65 equal 1
Row sums for age 66 equal 1
Row sums for age 67 equal 1
Row sums for age 68 do not equal 1
Row sums for age 69 do not equal 1


In [162]:
for age in age_range:
    if age in p:
       # Normalize rows of the NumPy array
        row_sums = p[age].sum(axis=1, keepdims=True)  # Sum of each row
        p[age] = np.where(row_sums > 0, p[age] / row_sums, np.ones_like(p[age]) / p[age].shape[1])

for age in age_range:
    if not np.allclose(p[age].sum(axis=1), 1):
        print(f"Row sums for age {age} do not equal 1")
    else:
        print(f"Row sums for age {age} equal 1")


Row sums for age 50 equal 1
Row sums for age 51 equal 1
Row sums for age 52 equal 1
Row sums for age 53 equal 1
Row sums for age 54 equal 1
Row sums for age 55 equal 1
Row sums for age 56 equal 1
Row sums for age 57 equal 1
Row sums for age 58 equal 1
Row sums for age 59 equal 1
Row sums for age 60 equal 1
Row sums for age 61 equal 1
Row sums for age 62 equal 1
Row sums for age 63 equal 1
Row sums for age 64 equal 1
Row sums for age 65 equal 1
Row sums for age 66 equal 1
Row sums for age 67 equal 1
Row sums for age 68 equal 1
Row sums for age 69 equal 1


/var/folders/px/brv0p26n7szcpwsq7dm55yy00000gn/T/ipykernel_34470/4277690575.py:5: RuntimeWarning: invalid value encountered in divide
  p[age] = np.where(row_sums > 0, p[age] / row_sums, np.ones_like(p[age]) / p[age].shape[1])


In [163]:
print(p[50])

[[0.19459459 0.61621622 0.10810811 ... 0.         0.         0.        ]
 [0.01796407 0.28143713 0.51497006 ... 0.         0.         0.        ]
 [0.         0.03389831 0.3559322  ... 0.         0.         0.        ]
 ...
 [0.         0.         0.         ... 0.32515337 0.07361963 0.        ]
 [0.         0.         0.         ... 0.47368421 0.23976608 0.        ]
 [0.         0.         0.         ... 0.13815789 0.77631579 0.        ]]


#### Consider the retirement model given by:

$$
V(x_t,\varepsilon_t , \theta) = \max_d\in \{D(x_t)\} \big\{ u(x,d) + \varepsilon_d + \beta
\underbrace{\int_{X} \int_{\Omega} V(x',\varepsilon') \pi(x'|x,d) q(\varepsilon'|x') dx' d\varepsilon' }_{EV(x,d)} \big\}
$$

$$
V(x_t,\varepsilon_t , \theta) = \max_{d\in \{0,1\}} \big\{ u(x,d) + \varepsilon_d + \beta
\underbrace{\int_{X} \int_{\Omega} V(x',\varepsilon') \pi(x'|x,d) q(\varepsilon'|x') dx' d\varepsilon' }_{EV(x,d)} \big\}
$$

Where $ \varepsilon $ is extreme value Type I distribued and utility is given by:

$$
u(x,d)=\left \{
\begin{array}{ll}
    -RC-c(0,\theta_1) & \text{if }d=\text{replace}=1 \\
    -c(x,\theta_1) & \text{if }d=\text{keep}=0
\end{array} \right.
$$

Here

- $ RC $ = replacement cost  
- $ c(x,\theta_1) $ = cost of maintenance with preference parameters $ \theta_1 $  




We get our transition probability vector, that is the transition probabilities stated in the setup, which is an array. Then we sum all probabilities and deduct from 1, leaving us with the probability for passing onto the last state, let's say we state probabilities of staying in state 0, moving to 1 or 2, but this does not sum to 1, then probability of moving to state 3 will be 1 - sum of all the stated probabilities.
    
- "Then we initialize the state transition matrix as a N x N matrix, with zeros.\n",

- "We loop over each row of the transition matrix:\n",
1. for iteration in range n, we check if the probability vector fits entirely in the row, if that is the case, we insert that at point i until the end of the p vector, imagine p vector is 4 x 1 and matrix is 12 x 12 (range 0 to 11), in the first iteration we then insert the matrix in row i (0), column i (0) and to column 3. in next iteration we interst p vector from row 1, column 1 to 4 and so forth.\n",
    "When the vector no longer fits, we insert in row (i) and colum (i) until the end, but the probabilities are summed \"backwards\". lets say we are in iteration 9, we then insert vector in row 9, first item in vector is inserted in column 9, second item in 10, and third and fourth are summed in column 11.\n",
    "\n",
    "Then we create transition probaiblity if engine is replaced as this places us back in the state 0, we create the N x N matrix, but with the probability vector for being in state 0.\n",
    "\n",
    "\n",

### Zurcher.setup\n",
    "When setting up the problem we go through multiply stepts\n",
    "a. we setup the parameters, first we define a gridspace by setting the n (number of gridpoints) and a max (in this case the max value of mileage) \n",
    "Then we setup the structural parameters:\n",
    "we set the transition probabilities (p), the replacement cost (RC), the cost (maintenance cost) parameter (c) and the discount factor (beta)\n",
    "\n",
    "b. in a. we set some placeholder values for each parameter, then we use the kwargs to update these values to the ones we specify for the baseline parameters\n",
    "\n",
    "c. we then call the function that creates the grid based on parameters\n",
    "\n",
    "\n",
    
    
    
### Zurcher.create_grid\n",
    "first, we create the mileage grid, that is a range that runs from 0 to n (the number of gridpoints we specified in the setup)\n",
    "\n",
    "second, we create a cost function, that is by multiplying the maintenance cost (c) with the mileage grid and some scaling parameter (in this case 0.001). So we have a function of maintenance cost that increases as mileage goes up.\n",
    "\n",
    "Third, with the mileage grid and the costfunction we find the state transition\n",
    "\n",
    "\n",

### Zurcher.state_transition\n",
    "We get our transition probability vector, that is the transition probabilities stated in the setup, which is an array. Then we sum all probabilities and deduct from 1, leaving us with the probability for passing onto the last state, let's say we state probabilities of staying in state 0, moving to 1 or 2, but this does not sum to 1, then probability of moving to state 3 will be 1 - sum of all the stated probabilities.\n",
    "\n",
    "Then we initialize the state transition matrix as a N x N matrix, with zeros.\n",
    "\n",
    "We loop over each row of the transition matrix:\n",
    "1. for iteration in range n, we check if the probability vector fits entirely in the row, if that is the case, we insert that at point i until the end of the p vector, imagine p vector is 4 x 1 and matrix is 12 x 12 (range 0 to 11), in the first iteration we then insert the matrix in row i (0), column i (0) and to column 3. in next iteration we interst p vector from row 1, column 1 to 4 and so forth.\n",
    "When the vector no longer fits, we insert in row (i) and colum (i) until the end, but the probabilities are summed \"backwards\". lets say we are in iteration 9, we then insert vector in row 9, first item in vector is inserted in column 9, second item in 10, and third and fourth are summed in column 11.\n",
    "\n",
    "Then we create transition probaiblity if engine is replaced as this places us back in the state 0, we create the N x N matrix, but with the probability vector for being in state 0.\n",
    "\n",
    "\n",
    
    
    
### Zurcher.bellman\n",
    "First we find the value of keeping, that is minus the maintenance cost + dot product of the expected value function from previous iteration (ev0) and the transition matrix of not replacing engine discounted at beta.\n",
    "Thus it can be seen as the immediate cost of keeping engine + the discounted expected future value of keeping the enginge. This is a n x 1 matrix.\n",
    "\n",
    "Second we find the value of replacing, that is the replacement cost and the maintenance cost in period 0 + the discounted expected future value of replacing the engine (beta times the dot product of the transition matrix of replacing and the expected value function from the previous iteration) This is a 1 x 1 matrix.\n",
    "\n",
    "After calculating value of keeping and value of replacing we evaluate and find MaxV:\n",
    "that is using np.max to find the maximum value between value of keeping and value of replacing\n",
    "\n",
    "We then compute the expected value over the two choices (keep or replace), that is logsum to handle expectation over unobserved states. It accounts for unobserved randomness in decision making process.\n",
    "That is the sum of the three: MaxV, the log sum of the exponentials of difference between value of keeping and maxV and difference between value of replacing and maxV (np.log(np.exp(value_keep - maxV) - np.exp(value_replace - maxV)))\n",
    "\n",
    "we then set the expected value function (ev1) equal to the logsum, that is updating expected value function after applying the bellman operator. \n",
    "\n",
    "In the bellman function we have the output parameter that controls what the bellman function returns.\n",
    "\n",
    "If output = 1 this returns the expected value function (ev1)\n",
    "\n",
    "If output = 2 the function returns expected value function (ev1) and choice probability of keeping enginge (pk)\n",
    "\n",
    "we compute choice probability of keeping engine\n",
    "        pk = 1/(1+np.exp(value_replace-value_keep))       \n",
    "\n",
    "and return ev1 and pk\n",
    "\n",
    "then compute derivative of the bellman operator by calling dbellman function: dev1 = self.dbellman(pk)\n",
    "\n",

### Zurcher.dbellman\n",
    "This function computes the derivstive of the bellman operator\n",
    "\n",
    "1 we set the derivative matrix, dev1, a N x N matrix of zeros, n being the number of states in the model. Each element of dev1[i, j] represents bellman operator for state i changes with respect to the expected value function for state j.\n",
    "\n",
    "2 We loop over the choices d==0 keep engine and d==1 replace engine.\n",
    "\n",
    "if d==0, the transition probability matrix, P, is set to P1 (prob. matrix for keeping engine) and choice probability is set to pk\n",
    "if d==1, the transition probability matrix, P, is set to p2 (prob. matrix for replacing) and choice probabilbity is set to 1 - pk.\n",
    "\n",
    "Then update derivative matrix, dev1 += self.beta * choice_prob.reshape(-1,1) * P\n",
    "- += : operator that adds value to existing variable\n",
    "- self.beta: discount factor which scales future value\n",
    "- choice_prob.reshape(-1,1): reshape choice probabilities from the loop into a N x 1 matrix\n",
    "- P : the transition probability matrix for the current choice\n",
    " \n"
   ]